In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import pickle

In [39]:
df = pd.read_pickle('../data/final_data/data.pkl')

In [40]:
test = pd.read_pickle('../data/final_data/test.pkl')

In [41]:
feature_cols = [c for c in df.columns
                if c not in ["item_cnt_month", "date_block_num",
                              "avg_price", "item_avg_cnt",
                              "shop_avg_cnt", "cat_avg_cnt"]]

In [42]:
X_test = test[feature_cols]

In [10]:
df.columns

Index(['shop_id', 'item_id', 'date_block_num', 'item_cnt_month', 'avg_price',
       'item_cnt_month_lag_1', 'item_cnt_month_lag_2', 'item_cnt_month_lag_3',
       'item_cnt_month_lag_6', 'item_cnt_month_lag_12', 'item_avg_cnt',
       'item_avg_cnt_lag_1', 'item_avg_cnt_lag_2', 'item_avg_cnt_lag_3',
       'shop_avg_cnt', 'shop_avg_cnt_lag_1', 'shop_avg_cnt_lag_2',
       'shop_avg_cnt_lag_3', 'item_cnt_month_rmean_3',
       'item_cnt_month_rmean_6', 'item_cnt_month_rmean_12', 'trend_1_2',
       'trend_1_12', 'item_category_id', 'cat_avg_cnt', 'cat_avg_cnt_lag_1',
       'avg_price_lag_1', 'month', 'year'],
      dtype='str')

In [30]:
df.shape

(10913804, 30)

In [31]:
X_test.shape

(214200, 25)

In [32]:
test.shape

(214200, 31)

In [45]:
extra = df[df["date_block_num"] == 33][
    ["shop_id", "item_id", "item_avg_cnt", "shop_avg_cnt", "item_category_id", "cat_avg_cnt"]
].drop_duplicates(subset=["shop_id", "item_id"])

test = test.merge(extra[["shop_id", "item_id", "item_avg_cnt", "shop_avg_cnt"]], 
                  on=["shop_id", "item_id"], how="left")
test = test.merge(extra[["item_category_id", "cat_avg_cnt"]].drop_duplicates("item_category_id"),
                  on="item_category_id", how="left")

In [46]:
item_lag1_global = (df[df.date_block_num == 33]
                    .groupby("item_id")["item_cnt_month"]
                    .sum().reset_index()
                    .rename(columns={"item_cnt_month": "item_lag1_global"}))

item_active = (df[df.date_block_num.isin([31,32,33])]
               .groupby("item_id")["item_cnt_month"]
               .apply(lambda x: (x > 0).sum()).reset_index()
               .rename(columns={"item_cnt_month": "item_active_months"}))


df = df.merge(item_lag1_global, on="item_id", how="left")
df = df.merge(item_active, on="item_id", how="left")


test = test.merge(item_lag1_global, on="item_id", how="left")
test = test.merge(item_active, on="item_id", how="left")

In [47]:
feature_cols = [c for c in df.columns 
                if c not in ["item_cnt_month", "date_block_num", "avg_price", "year"]]

In [49]:
X_train = df[(df.date_block_num >= 12) & (df.date_block_num < 33)][feature_cols].copy()
y_train = df[(df.date_block_num >= 12) & (df.date_block_num < 33)]["item_cnt_month"]
X_val   = df[df.date_block_num == 33][feature_cols].copy()
y_val   = df[df.date_block_num == 33]["item_cnt_month"]
X_test  = test[feature_cols].copy()

---

## catboost

---

In [50]:
from catboost import CatBoostRegressor

In [51]:
X_train_cat = X_train.copy()
X_val_cat   = X_val.copy()
X_test_cat  = X_test.copy()

In [52]:
for col in ["shop_id", "item_id", "item_category_id", "month"]:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_val_cat[col]   = X_val_cat[col].astype(str)
    X_test_cat[col]  = X_test_cat[col].astype(str)

In [54]:
X_train_cat.drop('trend_1_12', axis=1, inplace=True)
X_val_cat.drop('trend_1_12', axis=1, inplace=True)
X_test_cat.drop('trend_1_12', axis=1, inplace=True)

In [ ]:
model_cat = CatBoostRegressor(
    iterations=300,
    learning_rate=0.1,
    depth=5,
    random_seed=42,
    eval_metric="RMSE",
    early_stopping_rounds=30,
    verbose=100,
)

In [ ]:
model_cat.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=["shop_id", "item_id", "item_category_id", "month"],
)

In [201]:
val_pred_cat = model_cat.predict(X_val).clip(0, 20)
rmse_cat = mean_squared_error(y_val, val_pred_cat)
print(f"CatBoost Validation RMSE: {(rmse_cat)**0.5:.4f}")

CatBoost Validation RMSE: 0.6565


In [112]:
val_pred_cat = model_cat.predict(X_val).clip(0, 20)
rmse_cat = mean_squared_error(y_val, val_pred_cat)
print(f"CatBoost Validation RMSE: {(rmse_cat)**0.5:.4f}")

CatBoost Validation RMSE: 0.6604


In [ ]:
model_cat.get_feature_importance(prettified=True)

In [203]:
pred_cat = model_cat.predict(X_test).clip(0, 20)
pd.DataFrame({"ID": test["ID"], "item_cnt_month": pred_cat}).to_csv("../data/submission/cat4_submission.csv", index=False)

## Private Score 1.02762 ;(

---